####  Load and convert date columns

In [2]:
# Cell 1: Load data and fix date columns

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
orders = pd.read_csv('../data/olist_orders_dataset.csv')
items = pd.read_csv('../data/olist_order_items_dataset.csv')
payments = pd.read_csv('../data/olist_order_payments_dataset.csv')
reviews = pd.read_csv('../data/olist_order_reviews_dataset.csv')
customers= pd.read_csv('../data/olist_customers_dataset.csv')
products = pd.read_csv('../data/olist_products_dataset.csv')
sellers = pd.read_csv('../data/olist_sellers_dataset.csv')
cat_map = pd.read_csv('../data/product_category_name_translation.csv')
# Convert all date columns to datetime
date_cols = [
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date'
]
for col in date_cols:
 orders[col] = pd.to_datetime(orders[col], errors='coerce')
print('Date columns converted.')
print(orders[date_cols].dtypes)


Date columns converted.
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


#### Filter to delivered orders only and merge all tables


In [3]:
# Cell 2: Filter + merge into master dataframe
# Keep only delivered orders
delivered = orders[orders['order_status'] == 'delivered'].copy()
print(f'Delivered orders: {len(delivered):,} out of {len(orders):,}')
# Drop rows where delivery date is missing
delivered = delivered.dropna(subset=['order_delivered_customer_date'])
print(f'After dropping missing delivery dates: {len(delivered):,}')
# MERGE STEP BY STEP
# Step 1: orders + customers
df = delivered.merge(customers, on='customer_id', how='left')
print(f'After customers merge: {df.shape}')
# Step 2: + order_items
df = df.merge(items, on='order_id', how='left')
print(f'After items merge: {df.shape}')
# Step 3: + products
df = df.merge(products, on='product_id', how='left')
print(f'After products merge: {df.shape}')
# Step 4: + category translation (English names)
df = df.merge(cat_map, on='product_category_name', how='left')
print(f'After category merge: {df.shape}')
# Step 5: + sellers
df = df.merge(sellers, on='seller_id', how='left',
suffixes=('_customer','_seller'))
print(f'After sellers merge: {df.shape}')
# Step 6: + payments (aggregate first to avoid duplicates)
pay_agg = payments.groupby('order_id').agg(
 payment_type = ('payment_type', lambda x: x.mode()[0]),
 payment_value = ('payment_value', 'sum'),
 payment_installments = ('payment_installments', 'max')
).reset_index()
df = df.merge(pay_agg, on='order_id', how='left')
print(f'After payments merge: {df.shape}')
# Step 7: + reviews (take first review per order)
rev_agg = reviews.groupby('order_id').agg(
 review_score = ('review_score', 'mean')
).reset_index()
df = df.merge(rev_agg, on='order_id', how='left')
print(f'After reviews merge: {df.shape}')


Delivered orders: 96,478 out of 99,441
After dropping missing delivery dates: 96,470
After customers merge: (96470, 12)
After items merge: (110189, 18)
After products merge: (110189, 26)
After category merge: (110189, 27)
After sellers merge: (110189, 30)
After payments merge: (110189, 33)
After reviews merge: (110189, 34)


####  Engineer new features


In [4]:
# Cell 3: Feature engineering
# Delivery performance features
df['delivery_days'] = (
 df['order_delivered_customer_date'] - df['order_purchase_timestamp']
).dt.days
df['estimated_days'] = (
 df['order_estimated_delivery_date'] - df['order_purchase_timestamp']
).dt.days
df['days_early_late'] = df['estimated_days'] - df['delivery_days']
# Positive = arrived early, Negative = arrived late
df['on_time'] = df['delivery_days'] <= df['estimated_days']
# Time features
df['order_year'] = df['order_purchase_timestamp'].dt.year
df['order_month'] = df['order_purchase_timestamp'].dt.month
df['order_quarter'] = df['order_purchase_timestamp'].dt.quarter
df['order_dow'] = df['order_purchase_timestamp'].dt.dayofweek
df['order_hour'] = df['order_purchase_timestamp'].dt.hour
df['order_yearmonth'] = df['order_purchase_timestamp'].dt.to_period('M').astype(str)
df['order_purchase_timestamp'].dt.to_period('M').astype(str)
# Revenue features
df['total_item_value'] = df['price'] + df['freight_value']
# Category: fill missing with Unknown
df['product_category_name_english'] = (
 df['product_category_name_english']
 .fillna(df['product_category_name'])
 .fillna('Unknown')
)
# Remove extreme outliers in delivery_days (likely data errors)
df = df[(df['delivery_days'] > 0) & (df['delivery_days'] < 180)]
print(f'Master dataset shape: {df.shape}')
print()
print('New features created:')
new_cols = ['delivery_days','estimated_days','days_early_late','on_time',
 'order_year','order_month','order_quarter','order_dow',
 'order_hour','order_yearmonth','total_item_value']
print(df[new_cols].describe())

Master dataset shape: (110155, 45)

New features created:
       delivery_days  estimated_days  days_early_late     order_year  \
count  110155.000000   110155.000000    110155.000000  110155.000000   
mean       11.983323       23.439680        11.456357    2017.544578   
std         9.201712        8.822093         9.951147       0.503757   
min         1.000000        2.000000      -162.000000    2016.000000   
25%         6.000000       18.000000         7.000000    2017.000000   
50%        10.000000       23.000000        12.000000    2018.000000   
75%        15.000000       28.000000        17.000000    2018.000000   
max       175.000000      155.000000       146.000000    2018.000000   

         order_month  order_quarter      order_dow     order_hour  \
count  110155.000000  110155.000000  110155.000000  110155.000000   
mean        6.032064       2.355908       2.746185      14.748563   
std         3.230954       1.061321       1.963265       5.314862   
min         1.000

#### Save the master dataset

In [5]:
# Cell 4: Save clean master dataset
output_path = '../data/olist_master.csv'
df.to_csv(output_path, index=False)
print(f'Master dataset saved to: {output_path}')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print()
print('Null counts in key columns:')
key_cols = ['customer_state','product_category_name_english',
 'delivery_days','review_score','payment_type','price']
print(df[key_cols].isnull().sum())


Master dataset saved to: ../data/olist_master.csv
Shape: (110155, 45)
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_category_name_english', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'payment_type', 'payment_value', 'payment_installments', 'review_score', 'delivery_days', 'estimated_days', 'days_early_late', 'on_time', 'order_year', 'order_month', 'order_quarter', 'order_dow', 'order_hour', 'order_yearmonth', 'total_item_value']

Null counts in key columns:
c